In [ ]:
import pandas as pd

df = pd.read_parquet('data/preprocessed.parquet')

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.dtypes

In [ ]:
min_count = df['ClassLabel'].value_counts().min()
df['ClassLabel'].value_counts(), min_count

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=['ClassLabel'], axis=1)
y = df['ClassLabel']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=69)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

In [ ]:
type(X_train), type(X_test), type(y_train), type(y_test)

In [ ]:
numerical_cols = X.select_dtypes(include='number').columns.tolist()
numerical_cols

In [ ]:
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
from sklearn.pipeline import Pipeline as SciPipeline
from sklearn.preprocessing import MinMaxScaler
from sklearn.compose import ColumnTransformer
from sklearn.naive_bayes import GaussianNB
from lightgbm import LGBMClassifier
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier
import os

allocated_cpus = os.cpu_count() // 2

scaling_transformer = SciPipeline(
    steps=[("scaler", MinMaxScaler(feature_range=(0, 1)))]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("scaling_transformer", scaling_transformer, numerical_cols),
    ],
    remainder="passthrough",  # ensures other columns, if any, are not dropped
)

pipeline = ImbPipeline(
    [
        ("preprocessor", preprocessor),
        ("under", RandomUnderSampler(random_state=69, sampling_strategy="majority")),
        ("classifier", GaussianNB()),
    ]
)

# Define parameter grid for GridSearchCV
param_grid = [
    {
        "classifier": [
            LGBMClassifier(
                n_jobs=allocated_cpus,
            )
        ],
        "classifier__num_leaves": [25, 50],
        "classifier__n_estimators": [25, 50],
        "classifier__learning_rate": [0.1, 0.01],
    },
    {
        "classifier": [XGBClassifier(n_jobs=allocated_cpus)],
        "classifier__max_depth": [3, 6],
        "classifier__n_estimators": [25, 50],
        "classifier__learning_rate": [0.1, 0.01],
    },
    {"classifier": [GaussianNB()]},
]

In [ ]:
from collections import Counter

class_counts = Counter(y_train)

print(class_counts)

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold
kf = StratifiedKFold(n_splits=3, random_state=69, shuffle=True)

grid_search = GridSearchCV(pipeline, param_grid, cv=kf, scoring='f1_macro', n_jobs=os.cpu_count()//2, verbose=3)
grid_search.fit(X_train, y_train)

print("Best parameters found: ", grid_search.best_params_)
grid_search.best_estimator_

In [ ]:
import json
from sklearn.metrics import classification_report

y_pred = grid_search.best_estimator_.predict(X_test)
print(json.dumps(classification_report(y_test, y_pred, output_dict=True), indent=4))

In [ ]:
from sklearn.metrics import confusion_matrix
from matplotlib import pyplot as plt
import seaborn as sns

cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(6, 4))
sns.heatmap(cm, annot=True, fmt='d', cmap='coolwarm', cbar=False)
plt.title("Confusion Matrix")
plt.ylabel("Actual")
plt.xlabel("Predicted")
plt.show()

In [ ]:
import pickle
with open('models/best_model.pkl', 'wb') as f:
    pickle.dump(grid_search.best_estimator_, f)

In [ ]:
with open('models/best_model.pkl', 'rb') as f:
    loaded_model = pickle.load(f)


In [ ]:
data_to_predict = X_test.iloc[991].tolist()

In [ ]:
X_train.columns == X_test.columns

In [ ]:
data_to_predict_df = pd.DataFrame([data_to_predict], columns=X_test.columns)
loaded_model.predict(data_to_predict_df)

In [ ]:
y_test.iloc[991]

In [ ]:
X_test.columns